# RAG Vector DataBase

- Part A: Divide documents into CHUNKS
- Part B: Encode CHUNKS into VECTORS and put in Chroma
- Part C: Visualize our vectors

## Load Libraries

In [1]:
import os                     # Work with environment variables and file paths
import requests               # Send HTTP requests 
import glob                   # Find files using wildcard patterns
import subprocess             # Run external commands or shell processes

import tiktoken               # Tokenizer used for counting tokens in OpenAI models
import numpy as np            # Numerical computing

from dotenv import load_dotenv    # Load environment variables

from openai import OpenAI                          # For interacting with the OpenAI API
from langchain_openai import OpenAIEmbeddings      # Create embeddings using OpenAI models
from langchain_chroma import Chroma                # Chroma vector database integration for LangChain
from langchain_huggingface import HuggingFaceEmbeddings   # Create embeddings using HuggingFace models

from langchain_community.document_loaders import DirectoryLoader # load many files from a folder
from langchain_community.document_loaders import TextLoader # load text files into LangChain documents

from langchain_text_splitters import RecursiveCharacterTextSplitter  
# Splits large documents into smaller chunks for embedding

from sklearn.manifold import TSNE  
# t-SNE algorithm for reducing high-dimensional embeddings to 2D/3D for visualization

import plotly.graph_objects as go  
# Interactive plotting library

from IPython.display import Markdown, display  
# Display formatted Markdown output in Jupyter notebooks

from pathlib import Path  
# Modern object-oriented way to handle file paths

import re  
# Regular expressions for pattern matching in text

c:\Users\REDTECH\miniconda3\envs\llm-openvoice\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Environment Key

In [2]:
try:
    script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    script_dir = os.getcwd()

# Go up one level to the project folder
project_dir = os.path.dirname(script_dir)
env_path = os.path.join(project_dir, "env_keys", ".env")

# Access the variable
load_dotenv(dotenv_path=env_path)
openai_api_key = os.getenv("OPENAI_API_KEY")
print("API Key loaded:", openai_api_key is not None)

API Key loaded: True


## Ollma Initialize

In [3]:
subprocess.Popen("ollama serve", shell=True)

<Popen: returncode: None args: 'ollama serve'>

In [4]:
requests.get("http://localhost:11434").content

b'Ollama is running'

In [5]:
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(result.stdout)

NAME                ID              SIZE      MODIFIED    
qwen2.5-coder:7b    dae161e27b0e    4.7 GB    4 weeks ago    
llama3.2:latest     a80c4f17acd5    2.0 GB    4 weeks ago    



In [6]:
OLLAMA_API_URL = "http://localhost:11434/v1"
OPENAI_API_URL = "https://api.openai.com/v1"

## Initialize OpenAI Client for Ollma

In [7]:
openai = OpenAI(base_url=OPENAI_API_URL, api_key=openai_api_key)
ollama = OpenAI(base_url=OLLAMA_API_URL, api_key="None")

## Document Analysing

In [8]:
knowledge_base_path = "knowledge-base/**/*.md"

In [9]:
files = glob.glob(knowledge_base_path, recursive=True)
print(f"File in the Knowledge base : {len(files)}")

File in the Knowledge base : 76


In [10]:
entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total Characters in knowledge base: {len(entire_knowledge_base):,}")

Total Characters in knowledge base: 304,434


## Tokens

In [11]:
encoding = tiktoken.encoding_for_model("gpt-4.1-mini")
tokens = encoding.encode(entire_knowledge_base)
tokens_count = len(tokens)

print(f"Total tokens : {tokens_count:,}")

Total tokens : 63,555


## LangChain Loader

In [12]:
files = glob.glob("knowledge-base/*")

documents = []

for file_path in files:
    doc_type = os.path.basename(file_path)
    loader = DirectoryLoader(path=file_path, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for docs in folder_docs:
        docs.metadata['doc_type'] = doc_type
        documents.append(docs)

In [13]:
documents[2]

Document(metadata={'source': 'knowledge-base\\company\\culture.md', 'doc_type': 'company'}, page_content="# Insurellm Culture\n\n## Vision Statement\nTo revolutionize the insurance industry through innovative technology that makes insurance accessible, transparent, and effortless for everyone.\n\n## Mission Statement\nWe empower insurance providers and consumers with cutting-edge software solutions that streamline processes, enhance customer experiences, and drive meaningful connections in the insurance marketplace. By combining deep industry expertise with technological innovation, we're building the future of insurance.\n\n## Core Values\n\n### Innovation First\nWe challenge the status quo and embrace creative problem-solving. Our team is encouraged to experiment, take calculated risks, and push the boundaries of what's possible in insurance technology. We believe that breakthrough solutions come from curiosity, collaboration, and a willingness to learn from both successes and failur

## Character Text Split

In [14]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

In [15]:
Markdown(chunks[12].page_content)

## Employer Value Proposition

**Build the Future of Insurance Technology with Elite Talent**

At Insurellm, you'll join an elite team of 32 exceptional professionals transforming a traditional industry with cutting-edge technology. After our strategic restructuring in 2022-2023, we evolved from a 200-person startup to a lean, high-performing organization where every person makes a significant impact. With 32 active contracts across all eight product lines—from regional insurers to global reinsurance partners—each team member directly influences technology used by thousands of insurance professionals and millions of consumers. We offer:

## Make Vector Database and Store

In [16]:
# Encorder model for vector embeddings
embedding = OpenAIEmbeddings(model='text-embedding-3-large')
# embedding = HuggingFaceEmbeddings(model='all-MiniLM-L6-v2')

In [17]:
# Vector database
db_name = "vector_db"

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embedding).delete_collection()

vector_store = Chroma.from_documents(documents=chunks, embedding=embedding, persist_directory=db_name)
print(f"Vector store created with {vector_store._collection.count()} documents")

Vector store created with 413 documents


In [18]:
collections = vector_store._collection
some_embedding = collections.get(limit=1, include=['embeddings']).get("embeddings")

In [19]:
print(f"There are {(collections.count())} vectors with {some_embedding.shape[1]} dimensions in the vector store.")

There are 413 vectors with 3072 dimensions in the vector store.


In [20]:
result = collections.get(include=['embeddings', 'documents', 'metadatas'])

In [21]:
vectors = np.array(result['embeddings'])
documents = result['documents']
metadata = result['metadatas']
doc_type = [metadata['doc_type'] for metadata in metadata]

In [22]:
doc_types = ['products', 'employees', 'contracts', 'company']
colors = ['blue', 'green', 'red', 'orange']
color_map = dict(zip(doc_types, colors))

colors = [color_map[type] for type in doc_type]

In [23]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [24]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()